# Capítulo 5 - Simulação do Acoplamento Hidráulico-Mecânico
### Estudo da influência da membrana elástica na rede hidráulica
#### Disciplina: SME0602 — Motores Numéricos para Simulação em Engenharia
#### Professor: Roberto F. Ausas
#### Grupo 3: 
* #### Beatriz Cosimatti
* #### Cecilia Queiroz
* #### Gabriel Zago
* #### Matheus Buzzon
* #### Pedro Vale
* #### Victor Silva


In [ ]:
# Importações!!!

import sys
import os
import importlib

root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root not in sys.path:
    sys.path.insert(0, root)

import mechanic_hydraulic
importlib.reload(mechanic_hydraulic)
import env
importlib.reload(env)
from plotting import plot_relaxamento_problema3
config = env.CONFIG_MH

##### Para atingir um modelo computacional realista, é necessário, primeiramente, analisar a matemática que rege o problema. Portanto, como primeiro passo, deve-se deduzir a equação que molda o problema:

$$M\left(\frac{v^{n+1} - v^n}{\delta t}\right) + \left(\beta M + \frac{h}{2}U^T A^{-1}U\right)v^{n+1} + Kw^{n+1} = U^T A^{-1}b^{n+1}$$

##### As equações mecânicas e hidráulicas são resolvidas simultaneamente por meio de um sistema global:

$$
\mathbf{A}_{\mathrm{global}} \, \mathbf{x}^{n+1} = \mathbf{b}
$$

##### Que pode ser escrito como:

$$
\begin{bmatrix}
\frac{1}{\delta t}I & -I & 0 \\
K & \frac{1}{\delta t}M + D & -U^T \\
0 & \frac{h}{2}U & A
\end{bmatrix}
\begin{bmatrix}
w^{n+1} \\ v^{n+1} \\ p^{n+1}
\end{bmatrix}
=
\begin{bmatrix}
\frac{1}{\delta t}w^n \\
\frac{1}{\delta t}Mv^n \\
b^{n+1}
\end{bmatrix}
$$
 
$$\frac{1}{\delta t}w^{n+1} - v^{n+1} = \frac{1}{\delta t}w^n \tag{I}$$
 
$$Kw^{n+1} + \left(\frac{1}{\delta t}M + D\right)v^{n+1} - U^T p^{n+1} = \frac{1}{\delta t}Mv^n \tag{II}$$
 
$$\frac{h}{2}Uv^{n+1} + Ap^{n+1} = b^{n+1} \tag{III}$$
 
##### Passo 1: Isolar $p^{n+1}$ a partir da equação (III)
 
$$Ap^{n+1} = b^{n+1} - \frac{h}{2}Uv^{n+1}$$
 
$$p^{n+1} = A^{-1}\left(b^{n+1} - \frac{h}{2}Uv^{n+1}\right)$$
 
##### Passo 2: Isolar $w^{n+1}$ a partir da equação (I)
 
$$\frac{1}{\delta t}w^{n+1} = \frac{1}{\delta t}w^n + v^{n+1}$$
 
$$w^{n+1} = w^n + \delta t\, v^{n+1}$$
 
##### Passo 3: Substituir $p^{n+1}$ na equação (II)
 
$$Kw^{n+1} + \left(\frac{1}{\delta t}M + D\right)v^{n+1} - U^T\left[A^{-1}\left(b^{n+1} - \frac{h}{2}Uv^{n+1}\right)\right] = \frac{1}{\delta t}Mv^n$$
 
$$Kw^{n+1} + \frac{1}{\delta t}Mv^{n+1} + Dv^{n+1} - U^T A^{-1}b^{n+1} + \frac{h}{2}U^T A^{-1}Uv^{n+1} = \frac{1}{\delta t}Mv^n$$
 
##### Passo 4: Agrupar termos em $v^{n+1}$ e usar $D = \beta M$
 
$$\frac{1}{\delta t}Mv^{n+1} - \frac{1}{\delta t}Mv^n + \left(D + \frac{h}{2}U^T A^{-1}U\right)v^{n+1} + Kw^{n+1} = U^T A^{-1}b^{n+1}$$
 
$$M\left(\frac{v^{n+1} - v^n}{\delta t}\right) + \left(\beta M + \frac{h}{2}U^T A^{-1}U\right)v^{n+1} + Kw^{n+1} = U^T A^{-1}b^{n+1}$$

##### Essa formulação caracteriza um acoplamento monolítico, pois todas as variáveis do sistema são calculadas simultaneamente em cada passo temporal. 
 


![img problema 1](problema1.jpeg)

##### A Figura 1 apresenta a estrutura de esparsidade da matriz de amortecimento hidráulico equivalente (R), juntamente com a identificação dos graus de liberdade (GDLs) livres e restritos da membrana.

##### Observa-se que a matriz (R) possui dimensão 676x676, correspondente aos graus de liberdade da malha computacional 26x26. Entretanto, apenas os nós pertencentes ao interior da membrana circular participam efetivamente da dinâmica do sistema, enquanto os nós externos representam as condições de contorno impostas na borda da membrana.

##### A representação obtida evidencia a formação de um bloco denso na região correspondente aos graus de liberdade livres, enquanto as regiões associadas aos nós restritos permanecem praticamente vazias. Esse comportamento é consistente com a formulação matemática do problema, uma vez que o amortecimento hidráulico atua apenas sobre os graus de liberdade capazes de sofrer deslocamento.

##### Além disso, observa-se que a matriz equivalente apresenta uma estrutura significativamente mais densa do que as matrizes mecânicas originais. Esse resultado decorre diretamente da operação

$$
R = h^{2}U^{T}A^{-1}U,
$$

##### na qual a inversão da matriz hidráulica (A) introduz um acoplamento global entre os graus de liberdade da membrana. Dessa forma, uma variação de pressão provocada por uma deformação localizada propaga-se por toda a rede hidráulica, influenciando simultaneamente os demais pontos da membrana. Consequentemente, praticamente todos os graus de liberdade livres passam a interagir entre si, justificando o formato aproximadamente quadrado observado na região central da matriz.

##### Para a malha utilizada, a matriz apresenta aproximadamente 99 856 elementos não nulos, correspondendo a um fator de preenchimento (fill) de aproximadamente 21,9%. Embora essa porcentagem seja significativamente superior à das matrizes de rigidez e massa da membrana, a matriz ainda preserva uma quantidade considerável de elementos nulos, permitindo o uso eficiente de técnicas de armazenamento esparso.

##### A Figura 2 complementa essa análise ao apresentar a distribuição espacial dos graus de liberdade da membrana. Os nós destacados em verde representam os graus de liberdade livres, enquanto os nós em vermelho correspondem às regiões fixadas pelas condições de contorno. Observa-se que os nós livres reproduzem adequadamente a geometria circular da membrana, confirmando que a discretização espacial e a imposição das condições de contorno foram realizadas corretamente.

##### De modo geral, os resultados obtidos estão de acordo com o comportamento esperado para o sistema acoplado hidráulico-mecânico. A estrutura de esparsidade da matriz (R) evidencia o efeito global introduzido pelo fluido sobre a dinâmica da membrana, ao mesmo tempo em que confirma a consistência da implementação numérica utilizada na montagem do sistema acoplado.

##### Para se analisar o sistema ao longo do tempo, como existe um forte acoplamento entre o comportamento mecânico da membrana e o escoamento hidráulico, optou-se pela utilização de um método de integração temporal implícito, garantindo maior estabilidade numérica durante a simulação.

#### O Método de Euler Implícito
##### O método de Euler Implícito aproxima a derivada temporal utilizando uma diferença regressiva, avaliando a função no instante futuro t<sup>n+1. Assim: 
$$
\frac{d\mathbf{x}}{dt}
\approx
\frac{\mathbf{x}^{\,n+1}-\mathbf{x}^{\,n}}
{\Delta t}.
\tag{11}
$$

##### Substituindo essa aproximação na equacão diferencial obtém-se
$$
\mathbf{x}^{\,n+1}
=
\mathbf{x}^{\,n}
+
\Delta t\,
f(\mathbf{x}^{\,n+1},t^{\,n+1}),
\tag{13}
$$

##### Observa-se que as incógnitas do instante futuro aparecem em ambos os lados da equação, caracterizando um método implícito. Dessa forma, torna-se necessário resolver um sistema linear a cada passo de tempo para determinar o estado da solução.

##### O problema de evolução temporal foi resolvido considerando os passos de tempo adimensionais: 𝛿𝑡 = 0.00625, 0.0125, 0.025 e 0.05 dentro do intervalo de tempo [0, 12]. Para a membrana elástica, utilizamos ambas as discretizações espaciais de (51, 51) e (101, 101) e para a rede hidráulica, analisamos para diferentes pressões na entrada: 𝑝inlet = 5 × 10^3 Pa, 10^4 Pa e 2 × 10^4 Pa.

In [ ]:
simulador = mechanic_hydraulic.MechanicHydraulic(config)
todos_resultados = simulador.resolver_todos_cenarios(print_info=False)

##### Deslocamento Centro

![Deslocamento Centro P 5e+03](P2/deslocamento_centro_P_5.0e+03.png)

![Deslocamento Centro P 1e+04](P2/deslocamento_centro_P_1.0e+04.png)

![Deslocamento Centro P 2e+04](P2/deslocamento_centro_P_2.0e+04.png)


##### Perfil Membrana

![Perfil Membrana P 5e+03](P2/perfil_membrana_P_5.0e+03.png)

![Perfil Membrana P 1e+04](P2/perfil_membrana_P_1.0e+04.png)

![Perfil Membrana P 2e+04](P2/perfil_membrana_P_2.0e+04.png)

##### Potência

![Potência P 5e+03](P2/potencia_P_5.0e+03.png)

![Potência P 1e+04](P2/potencia_P_1.0e+04.png)

![Potência P 2e+04](P2/potencia_P_2.0e+04.png)

##### Pressão Outlet

![Pressão Outlet P 5e+03](P2/pressao_outlet_P_5.0e+03.png)

![Pressão Outlet P 1e+04](P2/pressao_outlet_P_1.0e+04.png)

![Pressão Outlet P 2e+04](P2/pressao_outlet_P_2.0e+04.png)

##### Vazão Outlet

![Vazão Outlet P 5e+03](P2/vazao_outlet_P_5.0e+03.png)
![Vazão Outlet P 1e+04](P2/vazao_outlet_P_1.0e+04.png)
![Vazão Outlet P 2e+04](P2/vazao_outlet_P_2.0e+04.png)

##### Volume Reservatório

![Volume Reservatório P 5e+03](P2/volume_reservatorio_P_5.0e+03.png)
![Volume Reservatório P 1e+04](P2/volume_reservatorio_P_1.0e+04.png)
![Volume Reservatório P 2e+04](P2/volume_reservatorio_P_2.0e+04.png)

##### A análise transiente do sistema acoplado eletromecânico-hidráulico foi realizada sob pressões nominais de entrada $P_{\text{inlet}} = 1.0 \times 10^4, 2.0 \times 10^4, 5.0 \times 10^3  \text{ Pa}$ para quatro larguras distintas de microcanais ($1000\,\mu\text{m}$, $1250\,\mu\text{m}$, $1500\,\mu\text{m}$ e $1750\,\mu\text{m}$). O comportamento transiente do acoplamento foi avaliado cruzando-se duas densidades de malha espacial ($51 \times 51$ e $101 \times 101$) com quatro passos de tempo adimensionais ($\delta t = 0.00625, 0.0125, 0.025 \text{ e } 0.05$) ao longo do intervalo de tempo normalizado $t \in [0, 12]$.

### 1. Dinâmica de Deflexão e Deslocamento da Membrana
Os gráficos de Deslocamento Vertical do Ponto Central e o Perfil Transiente de Deflexão revelam um comportamento oscilatório subamortecido característico de sistemas com acoplamento de segunda ordem (massa-mola-amortecedor). 

* Influência da Largura do Canal: À medida que a largura do canal aumenta de $1000\,\mu\text{m}$ para $1750\,\mu\text{m}$, observa-se um aumento expressivo na amplitude máxima de deflexão central da membrana (saltando de aproximadamente $0.00012\text{ m}$ para mais de $0.00016\text{ m}$). Além disso, canais mais largos reduzem drasticamente o amortecimento efetivo do sistema. Para o canal de $1000\,\mu\text{m}$, o transiente atinge o equilíbrio estável por volta de $t = 8$. Já para o canal de $1750\,\mu\text{m}$, fortes oscilações persistem até o final do intervalo de simulação ($t = 12$). Isso ocorre porque canais mais largos diminuem a restrição viscosa ao fluxo, reduzindo o amortecimento hidráulico exercido sobre a membrana.
* Morfologia da Deflexão: O perfil espacial de corte central ($Y=0$) valida as condições de contorno de engastamento/contorno nulo nas extremidades ($\pm 1.0$). A evolução temporal mostra a transição suave de repouso ($t=0.000$) até o perfil parabólico estabilizado nos tempos superiores ($t=9.600$ e $t=12.000$).

### 2. Comportamento Hidráulico no Nó de Descarga ($p_{\text{outlet}}$ e $q_{\text{outlet}}$)
As curvas de Pressão no Nó de Descarga e Vazão de Saída refletem de forma direta a pulsação mecânica induzida pela membrana sobre o fluido contido no reservatório.

* Pressão Transiente ($p_{\text{outlet}}$): No instante inicial ($t=0$), a súbita pressurização do sistema gera um pico agudo de pressão que se propaga pela rede. Em seguida, a pressão oscila em fase com o movimento da membrana. Notavelmente, a amplitude das oscilações de pressão diminui ligeiramente para canais mais largos devido à menor perda de carga localizada na descarga, tendendo à estabilização assimptótica em torno do valor de equilíbrio da rede.
* Vazão Transiente ($q_{\text{outlet}}$): A vazão apresenta comportamento altamente dinâmico, registrando valores inclusive *negativos* durante os vales de oscilação (especialmente perceptível nos canais de $1500\,\mu\text{m}$ e $1750\,\mu\text{m}$). Esse refluxo físico indica que, quando a membrana recua em sua oscilação ascendente, ela gera uma sucção transiente no reservatório, puxando o fluido de volta antes da próxima onda de compressão.

### 3. Volume Acumulado e Potência Consumida pelo Sistema
* Volume no Reservatório: O gráfico de Volume Acumulado atua como um integrador da vazão líquida ao longo do tempo. O volume acompanha os ciclos de expansão e contração da membrana. Nos canais mais estreitos ($1000\,\mu\text{m}$), o volume converge rapidamente para um patamar fixo estabilizado em torno de $0.8 \times 10^{-9}\text{ m}^3$. Nos canais mais largos, o volume experimenta grandes ciclos de esvaziamento parcial e enchimento, demorando mais para alcançar o estado estacionário.
* Potência Hidráulica: O perfil de Potência Consumida exibe picos sucessivos que decrescem exponencialmente ao longo do tempo, acompanhando a dissipação de energia por efeitos viscosos (amortecimento). O maior consumo energético ocorre invariavelmente no primeiro ciclo de deflexão ($t \approx 1$), onde o trabalho mecânico para tirar a membrana da inércia e vencer a resistência hidrodinâmica inicial exige a máxima potência do sistema.

### 4. Análise de Convergência Numérica
A análise de sensibilidade numérica revela importantes conclusões sobre a estabilidade do modelo computacional implementado:

* Efeito da Discretização Espacial (Malha): As curvas para as malhas $51 \times 51$ (linhas contínuas/tracejadas azuis) e $101 \times 101$ (linhas laranjas) mostram excelente concordância mútua. Isso comprova que a malha de $51 \times 51$ já é refinada o suficiente para garantir a independência de malha espacial, capturando a física do fenômeno com baixo custo computacional.
* Efeito da Largura do Canal: A largura do canal do sistema hidráulico desempenha um papel crítico na precisão do integrador transiente. Fica evidente em todos os gráficos que quanto maior a largura do canal, mais o sistema demora para convergir, ou seja, a largura do canal tem uma relação de proporcionalidade inversa à intensidade do amortecimento geral do sistema.

##### A partir das configurações obtidas ao final da simulação do item anterior, estudamos o cenário em que a pressão de entrada 𝑝 inlet decaiu instantaneamente para zero. O sistema continuou evoluindo a partir desse estado transiente até atingir um novo estado de equilíbrio estático

In [ ]:
#Roda o caso base uma vez com as restrições do enunciado para gerar o ponto de partida (estado inflado)
estado_inflado_ex2 = simulador.resolver_caso_base(
    N=(51, 51),
    dt=0.025,
    tempo_final=config["TIME_END"], 
    pressao_inlet=config["INLET_PRESSURE"], 
    largura_canal=config["CHANNEL_WIDTH"],
    print_info=False
)

resultado_problema3 = simulador.resolver_relaxamento(
    estado_inicial_ex2=estado_inflado_ex2,
    dt=0.025,
    tempo_final=12.0, 
    largura_canal=config["CHANNEL_WIDTH"],
    print_info=False
)
plot_relaxamento_problema3(resultado_problema3)

![Resultado problema 3](Figure_cap5_p3(versao2).png)

##### O estudo do comportamento dinâmico do sistema sob a condição de relaxamento numérico foi conduzido partindo-se do estado transiente final do ensaio anterior. Avaliou-se o comportamento do acoplamento mecânico-hidráulico após a remoção instantânea da excitação de carga, impondo-se a pressão de entrada $p_{\text{inlet}} = 0 \text{ Pa}$ para o intervalo de tempo normalizado $t \in [0, 12]$. Conforme restrição do modelo, utilizou-se a malha espacial estável de $51 \times 51$ pontos acoplada ao passo de tempo fixado em $\delta t = 0.025$.

### 1. Dinâmica de Relaxamento e Deflexão da Membrana
O gráfico de Deflexão no Ponto Central traduz o fenômeno mecânico de restauração elástica da membrana em um meio fluido viscoso.

* Efeito Elástico de Restituição: No instante $t = 0$, a membrana inicia o processo contendo uma energia potencial elástica acumulada (deflexão inicial em torno de $7.5 \times 10^{-5} \text{ m}$). Com a queda abrupta da pressão de entrada para zero, a força de restituição elástica da membrana força seu retorno à posição de equilíbrio plano ($w = 0$).
* Comportamento Subamortecido: O relaxamento não ocorre de forma puramente monotônica. A energia cinética adquirida pela estrutura durante o retorno faz com que ela ultrapasse a linha neutra, atingindo uma deflexão negativa (vale em $t \approx 1.2$) antes de oscilar de forma amortecida. O amortecimento viscoso do fluido dissipa progressivamente essa energia mecânica, convergindo estaticamente para a deflexão nula ($w = 0$) após $t = 6$.

### 2. Resposta Hidrodinâmica Transiente ($p_{\text{outlet}}$ e $q_{\text{outlet}}$)
A perda repentina de carga na entrada altera drasticamente o gradiente de pressões e os fluxos de massa no interior do reservatório.

* Queda Inversiva de Pressão ($p_{\text{outlet}}$): No exato instante do corte de energia ($t=0$), a pressão no nó de descarga cai instantaneamente a quase zero devido ao alívio da linha, mas imediatamente volta a subir para cerca de $3000 \text{ Pa}$ impulsionada pelo primeiro movimento descendente de compressão da membrana. Ela acompanha fielmente a frequência de oscilação amortecida da estrutura mecânica até estabilizar-se em zero (equilíbrio hidrostático).
* Inversão da Vazão de Saída ($q_{\text{outlet}}$): Como a membrana inicialmente se move para mitigar a deflexão acumulada, ela empurra o fluido para fora no primeiro ciclo. Entretanto, no movimento de rebote (quando a deflexão cruza a linha zero e torna-se negativa), ocorre uma forte inversão no vetor de velocidade do escoamento. O gráfico registra um pico de vazão negativa expressivo ($\approx -2.2 \times 10^{-5} \text{ m}^3/\text{s}$), evidenciando um refluxo físico onde o reservatório atua sugando fluido da saída para preencher o espaço volumétrico expandido pela membrana.

### 3. Volume do Reservatório e Potência Dissipada
* Volume do Reservatório de Fluido: O gráfico de volume acumulado mimetiza perfeitamente a oscilação da deflexão central. Ele inicia em um estado preenchido ($\approx 7.5 \times 10^{-10} \text{ m}^3$), atinge valores negativos de volume relativo (esvaziamento severo induzido pela deflexão negativa da membrana) e estabiliza-se no patamar nulo, indicando que o reservatório retornou ao seu volume nominal de repouso sem retenção líquida residual.
* Potência Consumida/Dissipada: Por não haver mais acionamento ou trabalho útil sendo injetado pela bomba ($p_{\text{inlet}} = 0$), a curva de potência neste problema representa estritamente a taxa de dissipação de energia viscosa. O sistema experimenta um pico de dissipação energética inicial violento ($\approx 0.07 \text{ W}$) em $t \approx 0.5$, associado à alta velocidade com que o fluido é forçado a se mover pelos microcanais no primeiro ciclo de relaxamento. À medida que as oscilações perdem amplitude, a potência dissipada cai exponencialmente a zero, confirmando o esgotamento da energia mecânica total do sistema.

##### Depois, medimos a frequência de oscilação transiente do sistema considerando...

## 1.Pressão nula na entrada:

In [ ]:
config["N"] = (101, 101) 

solver_p4 = mechanic_hydraulic.MH_Problema4(config)
solver_p4.resolver_P4(dt=0.0125, tempo_final=12.0)

![3 Modo Membrana](condição_inicial_3_modo_membrana.png)
![Deslocamento central vc lateral](deslocamento_no_central_vs_lateral.png)
![Pressao Outlet](pressão_P_outlet.png)

##### O Problema 4 propõe o estudo da oscilação livre do sistema inicializado a partir do terceiro modo fundamental de vibração da membrana, sob condição de escoamento livre ($\hat{\beta} = 0$, $H_k = 2000\,\mu\text{m}$ e $p_{\text{inlet}} = 0$). O objetivo é avaliar a capacidade do modelo acoplado de emular frequências naturais estruturais e verificar o acoplamento volumétrico residual.

### 1. Natureza Antissimétrica do Terceiro Modo e a Linha Nodal
O primeiro gráfico ilustra o campo espacial da condição inicial imposta. O terceiro modo de uma membrana circular é antissimétrico (geralmente associado ao modo indexado por coordenadas angulares e radiais, apresentando uma linha nodal reta que corta o centro do domínio).

* Comportamento do Nó Central: No segundo gráfico, observa-se que o deslocamento transiente do nó central (linha tracejada vermelha) permanece rigorosamente nulo ($\hat{w} = 0$) ao longo de todo o tempo. Isso é fisicamente exato: por estar posicionado exatamente sobre a linha nodal geométrica do terceiro modo, o centro da membrana não experimenta translação vertical durante a oscilação livre.
* Comportamento do Nó Lateral: O nó posicionado fora do eixo nodal (linha contínua azul) executa uma oscilação harmônica perfeita e subamortecida. Como as perdas viscosas foram minimizadas ($\hat{\beta}=0$ e canais largos), o amortecimento é sutil, provocado apenas pela inércia residual do fluido, permitindo medir a frequência natural com precisão. O erro obtido de apenas $0.62\%$ na malha $101 \times 101$ valida de forma incontestável a precisão do método de diferenças finitas e do integrador temporal implementados.

### 2. Ausência de Pressão no Outlet ($\hat{p} \approx 10^{-16}$)
O terceiro gráfico apresenta um resultado que mostra que a pressão no nó de descarga é nula (ordem de grandeza de $10^{-16}$ Pa é apenas ruído numérico de ponto flutuante).

* Explicação Física do Cancelamento Volumétrico: Por ser um modo puramente antissimétrico, enquanto uma metade da membrana está subindo (gerando sucção e volume negativo no reservatório), a outra metade está descendo exatamente com a mesma amplitude e velocidade (gerando compressão e volume positivo).
* Variação de Volume Líquido Nula: O efeito integrado de variação volumétrica interna da membrana sobre o fluido é rigorosamente zero ($\Delta V_{total} = 0$). Como não há variação de volume líquido no reservatório a cada passo de tempo, o fluido não é compelido a escoar em direção à saída. Consequentemente, a vazão $q_{\text{outlet}}$ e o gradiente de pressão transiente na descarga $\hat{p}_{\text{outlet}}$ anulam-se analiticamente.



### 2. Pressão na entrada: $$p_{\text{inlet}}(t) = 5000\cos(\omega_3 t) \ \text{Pa}$$

In [ ]:
solver_p5 = mechanic_hydraulic.MH_Problema5(config)
solver_p5.resolver_P5();

![Deslocamento Transiente](Figure_1.png)
![Pressao No descarga](Figure_2.png)


##### O Problema 5 explora a resposta transiente do sistema multifísico quando submetido a um carregamento harmônico na entrada, cuja frequência de excitação coincide exatamente com a frequência angular do terceiro modo fundamental da membrana (mostrado no problema anterior) ($p_{\text{inlet}}(t) = 5000 \cos(\omega_3 t)\text{ Pa}$). O estudo visa investigar a interação entre o forçamento hidráulico externo e a resposta mecânica da estrutura.

### 1. Quebra da Antissimetria e Ativação do Nó Central
No ensaio anterior (oscilação livre no terceiro modo), o nó central permaneça estagnado em zero devido à natureza puramente antissimétrica da condição inicial geométrica. Contudo, a introdução do carregamento harmônico altera completamente essa dinâmica:

* Deslocamento do Nó Central (Linha Vermelha): Diferente do Problema 4, o nó central agora deixa de ser nulo e passa a oscilar com grande amplitude (atingindo picos superiores a $\hat{w} = 2.0$). 
* Mecanismo Físico da Quebra de Simetria: A pressão harmônica imposta $p_{\text{inlet}}(t)$ atua de maneira uniforme sobre a área de captação da rede de microcanais que alimenta o reservatório. Um forçamento espacialmente uniforme é uma força simétrica, o que significa que ele possui forte acoplamento com o primeiro modo fundamental (modo simétrico/bombeador) da membrana. Portanto, embora a frequência de oscilação seja a do terceiro modo ($\omega_3$), o forçamento força a ativação de componentes espaciais simétricas, destruindo a linha nodal estática e forçando o centro da membrana a deslocar-se energeticamente.
* Fenômeno de Batimento e Modulação: O gráfico de deslocamento exibe uma modulação de amplitude (envoltória ondulatória). Esse comportamento é uma assinatura típica de sistemas acoplados onde a energia mecânica transita continuamente entre os modos induzidos pela condição inicial (Modo 3 antissimétrico) e os modos excitados pelo forçamento hidráulico simétrico.

### 2. Surgimento de Pressão Dinâmica no Nó de Descarga ($p_{\text{outlet}}$)
O resultado mais marcante da simulação é o comportamento da Pressão no Nó de Descarga, que contrasta drasticamente com a ausência total de oscilações verificada no Problema 4.

* De $10^{-16}\text{ Pa}$ para Escala Real ($\pm 6\text{ Pa}$): Como o centro da membrana agora se desloca (ativando o modo simétrico de bombeamento), o cancelamento volumétrico perfeito que existia no item anterior deixa de ocorrer. Agora, o movimento integrado da membrana gera uma variação real e líquida de volume ($\Delta V_{\text{total}} \neq 0$) no interior do reservatório a cada passo de tempo.
* Bombeamento Hidráulico: Essa variação volumétrica força o fluido a ser empurrado e puxado periodicamente através do microcanal de descarga, gerando um gradiente de pressão dinâmico real e bem definido no *outlet*, oscilando estavelmente em regime harmônico com amplitude de aproximadamente $6\text{ Pa}$.

A investigação dinâmica do sistema acoplado revelou a forte interdependência entre a deformação elástica da membrana e a resposta hidrodinâmica dos microcanais. Os ensaios transientes permitiram delinear de forma clara os limites físicos do modelo e estabelecer os critérios numéricos necessários para garantir a fidelidade do Gêmeo Digital.

Fisicamente, observou-se que a geometria dos microcanais dita o fator de amortecimento do sistema global. Canais estreitos ($1000\,\mu\text{m}$) restringem o escoamento, atuando como amortecedores eficazes que atenuam rapidamente os surtos de pressão transientes. Em contrapartida, canais largos ($1750\,\mu\text{m}$) reduzem a perda de carga viscosa e maximizam a deflexão estrutural, porém prolongam o regime oscilatório e induzem refluxos críticos ($q_{\text{outlet}} < 0$) no sistema. O ensaio de relaxamento ($p_{\text{inlet}} = 0$) corroborou essa dinâmica, demonstrando que a dissipação de energia viscosa nos canais é o mecanismo governante para o retorno seguro do sistema ao equilíbrio estático e repouso nominal.

Do ponto de vista analítico e numérico, a malha espacial de $51 \times 51$ pontos demonstrou excelente convergência e independência de refinamento. No entanto, o integrador temporal mostrou-se altamente sensível, exigindo passos de tempo estritos ($\delta t \le 0.0125$) para evitar a atenuação numérica artificial das ondas físicas. Por fim, a simulação do terceiro modo fundamental de vibração coroou a precisão matemática do solver ao registrar um erro de frequência marginal de apenas $0.62\%$. Esse cenário validou na prática o princípio de simetria do modelo: por se tratar de um modo puramente antissimétrico, ocorre um cancelamento volumétrico integral no reservatório, provando que o acoplamento estrutural pode ocorrer em regime de equilíbrio hidrostático nulo na descarga ($\hat{p}_{\text{outlet}} \approx 0$). Além disso, o comportamento do sistema em ressonância com a frequência do terceiro modo também se mostrou fisicamente preciso.